In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [ ]:
%config InlineBackend.figure_format = 'retina'

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"


# AbstractGraphPreprocessor Demo

Attention-derived base graphs from token arrays: train a small Transformer, build robust token-level edges via iterated MST + DP + consensus, and visualize one example.



In [ ]:
# Notebook bootstrap already added the repo root and sibling src directories to sys.path.
print("Repo root on sys.path:", repo_root)


In [ ]:
%time import numpy as np
%time import matplotlib.pyplot as plt
%time import networkx as nx
%time import torch

%time from abstractgraph_graphicalizer.attention import AbstractGraphPreprocessor, ImageNodeClusterer

np.random.seed(0)
_ = torch.manual_seed(0)

## Build a toy dataset

Each sample is a variable-length token matrix `(n_tokens, d_in)`.
Labels are generated by shifting token means (`A` ~ +0.5, `B` ~ -0.5)
so a shallow model can learn quickly.

In [ ]:
def make_dataset(n=40, d_in=16, len_lo=8, len_hi=16, shift=0.5, rng=None):
    rng = np.random.default_rng(0) if rng is None else rng
    X, y = [], []
    for _ in range(n):
        is_A = rng.random() < 0.5
        L = int(rng.integers(len_lo, len_hi))
        mu = shift if is_A else -shift
        x = rng.normal(loc=mu, scale=1.0, size=(L, d_in)).astype(np.float32)
        X.append(x)
        y.append('A' if is_A else 'B')
    return X, np.array(y)

X, y = make_dataset(n=40, d_in=16)
len(X), y[:5]

## Configure and train the preprocessor

We use a small Transformer and a simple label function for interpretation nodes (group size).
Optionally enable dataset-level node clustering if scikit-learn is available.

In [ ]:
# Optional clustering over pooled node embeddings (KMeans)
try:
    _clusterer = ImageNodeClusterer(n_clusters=8, random_state=0)
except Exception as e:
    print('Node clustering disabled:', e)
    _clusterer = None

pp = AbstractGraphPreprocessor(
    d_model=64, n_heads=4, num_layers=2, dim_feedforward=128, dropout=0.1,
    K_mst=3, alpha=5.0, beta=0.1, rho=0.3,  # loosen thresholds
    n_epochs=5, lr=1e-3, device="cpu",
    label_fn=lambda arr, idx: int(len(idx)),
    node_clusterer=_clusterer,
)


_ = pp.fit(X, y)
graphs = pp.transform(X)
graphs = pp.assign_node_cluster_labels(graphs)

n_nodes_total = sum(G.number_of_nodes() for G in graphs)
n_edges_total = sum(G.number_of_edges() for G in graphs)
print(f'Transformed {len(graphs)} graphs → total tokens={n_nodes_total}, total robust edges={n_edges_total}')


## Inspect and visualize one example

In [ ]:
i = 0
G = graphs[i]
print('Tokens:', G.number_of_nodes(), 'Robust edges:', G.number_of_edges())
co = G.graph.get('co_cluster')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# Left: co-clustering heatmap
ax = axes[0]
im = ax.imshow(co, cmap='viridis', vmin=0.0, vmax=1.0)
ax.set_title('Token Co-clustering')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xlabel('token idx')
ax.set_ylabel('token idx')

# Right: graph view
ax = axes[1]
pos = nx.spring_layout(G, seed=0) if len(G) > 0 else {}
clusters = [G.nodes[n].get('cluster_id', -1) for n in G.nodes()]
cmap = plt.cm.tab20
colors = [cmap((c % 20) / 20.0) if c is not None and c >= 0 else (0.6,0.6,0.6,1.0) for c in clusters]
sizes = [80 for _ in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=sizes, linewidths=0.5, edgecolors='k', ax=ax)
widths = [1.0 + 2.0*G.edges[u,v].get('weight', 0.0) for u,v in G.edges()]
nx.draw_networkx_edges(G, pos, width=widths, alpha=0.7, ax=ax)
ax.set_title('Graph (color=cluster_id, width=weight)')
ax.axis('off')
plt.tight_layout()
plt.show()

# Peek at first few nodes
for n in list(G.nodes())[:5]:
    data = G.nodes[n]
    emb_shape = tuple(np.asarray(data.get('embedding')).shape) if 'embedding' in data else None
    print({'node': int(n), 'cluster_id': data.get('cluster_id', None), 'embedding_shape': emb_shape})

# Visualize with AbstractGraph display utilities
from abstractgraph.graphs import AbstractGraph
from abstractgraph.display import display as ag_display
# Use cluster to color nodes (copied into 'label')
for n in G.nodes():
    G.nodes[n]['label'] = G.nodes[n].get('cluster_id', None)
AG = AbstractGraph(graph=G).create_default_image_node()
ag_display(AG, size=(6,5))
